# Phase 16K — independent full QLoRA training

This notebook **prepares** the 75-optimizer-step English-only and multilingual runs using the existing trainer. Training switches and Drive persistence default to `False`; Run all cannot start training. Use a Colab NVIDIA GPU runtime. The Phase 16H frozen data, Phase 16I full configurations, and Phase 16J CUDA smoke artifacts are prerequisites. No held-out evaluation is performed here. See `docs/training/phase16k_full_training_guide.md`.

## 1. GPU verification

In Colab, select **Runtime → Change runtime type → GPU** before running. This checks the assigned device and driver before installation.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

smi = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv,noheader,nounits",
    ],
    capture_output=True,
    text=True,
)
print(smi.stdout or smi.stderr)
if smi.returncode:
    raise RuntimeError("No NVIDIA GPU runtime is visible. Stop here and select a GPU runtime.")

## 2. Repository setup

Upload a freshly built Phase 16K allowlisted archive from your Mac. Extraction refuses links, unexpected paths, and an existing project directory. If using a private Git clone instead, copy the five frozen files at their exact relative paths before continuing; the frozen files are normally untracked.

In [ ]:
import tarfile

from google.colab import files

PROJECT = Path("/content/multilingual-llm-reasoning")
if PROJECT.exists():
    raise FileExistsError(f"{PROJECT} already exists; inspect it rather than overwriting")
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.endswith(".tar.gz")]
if len(archives) != 1:
    raise ValueError("Upload exactly one archive built with --phase16i-frozen")
with tarfile.open(archives[0], "r:gz") as archive:
    members = archive.getmembers()
    for member in members:
        parts = Path(member.name).parts
        if (
            not parts
            or parts[0] != PROJECT.name
            or ".." in parts
            or not (member.isfile() or member.isdir())
            or member.issym()
            or member.islnk()
        ):
            raise ValueError(f"Unsafe or unexpected archive member: {member.name}")
    archive.extractall(PROJECT.parent, filter="data")
assert (PROJECT / "configs/phase16i_english_smoke.yaml").is_file()
assert (PROJECT / "outputs/phase16i-training-preparation-v1/sft_training_manifest.json").is_file()
print("Project:", PROJECT)

## 3. Locked dependency installation

The existing installer creates an isolated Python 3.11.14 environment; it does not alter the notebook interpreter. Its hash-locked requirements include CUDA PyTorch, Transformers, PEFT, Accelerate, TRL, datasets, and bitsandbytes. Installation downloads CUDA packages, but does not download model weights.

In [ ]:
subprocess.run(["bash", "scripts/setup_colab.sh"], cwd=PROJECT, check=True)
PYTHON = PROJECT / ".venv-colab/bin/python"
ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
subprocess.run(
    [
        str(PYTHON),
        "-c",
        "import torch, transformers, peft, accelerate, trl, bitsandbytes; "
        "print('torch', torch.__version__, 'CUDA', torch.version.cuda, "
        "'available', torch.cuda.is_available()); "
        "print('transformers', transformers.__version__, 'peft', peft.__version__, "
        "'accelerate', accelerate.__version__, 'trl', trl.__version__, "
        "'bitsandbytes', bitsandbytes.__version__)",
    ],
    cwd=PROJECT,
    env=ENV,
    check=True,
)
print("Frozen environment:", PROJECT / ".venv-colab/environment.freeze.txt")

## 4. Frozen dataset integrity and model revision

The exact SHA-256 values below are the Phase 16H freeze. The model revision is checked against both YAML configs and the freeze manifest. A lightweight Hub metadata query checks access to that exact revision without downloading weights; the trainer later loads that revision.

In [ ]:
import hashlib

FREEZE = PROJECT / "outputs/phase16h-approved-sft-freeze-v1"
EXPECTED = {
    "english_only_sft_final.jsonl": (
        "6e59b5344db55eb84f87c9dc9059587fbd00dae0edcea967a76d37705b8ed881"
    ),
    "multilingual_sft_final.jsonl": (
        "9b5c813da240229a31a7e57d1bc8a8e4cb9e8a410a837c38b01db23202df2021"
    ),
}
for name, digest in EXPECTED.items():
    actual = hashlib.sha256((FREEZE / name).read_bytes()).hexdigest()
    print(name, actual)
    assert actual == digest, f"Frozen dataset changed: {name}"
freeze_manifest = json.loads((FREEZE / "dataset_freeze_manifest.json").read_text())
assert freeze_manifest["training_ready"] is True
assert all(
    freeze_manifest["dataset_files_sha256"][name] == digest for name, digest in EXPECTED.items()
)
REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"
for name in ("english_smoke", "multilingual_smoke", "english_lora", "multilingual_lora"):
    parsed = subprocess.run(
        [
            str(PYTHON),
            "-c",
            "import json,sys,yaml; print(json.dumps(yaml.safe_load(open(sys.argv[1]).read())))",
            str(PROJECT / f"configs/phase16i_{name}.yaml"),
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
        capture_output=True,
        text=True,
    )
    config = json.loads(parsed.stdout)
    assert config["model"] == "Qwen/Qwen2.5-3B-Instruct"
    assert config["revision"] == REVISION
assert freeze_manifest["model_revision"] == freeze_manifest["tokenizer_revision"] == REVISION
subprocess.run(
    [
        str(PYTHON),
        "-c",
        "from huggingface_hub import HfApi; "
        "import sys; "
        "r = HfApi().model_info(sys.argv[1], revision=sys.argv[2]); "
        "print('Resolved model SHA:', r.sha); "
        "assert r.sha == sys.argv[2]",
        "Qwen/Qwen2.5-3B-Instruct",
        REVISION,
    ],
    cwd=PROJECT,
    env=ENV,
    check=True,
)

## 5. Matched full-configuration comparison and CUDA preflight

The existing Phase 16I preflight checks GPU/VRAM, locked packages, bitsandbytes NF4, disk space, frozen hashes, tokenizer identity, and matched hyperparameters. The Phase 16K wrapper additionally validates both full configs and the 75-step budget without downloading weights. Stop on any blocker.

In [ ]:
PREFLIGHT = PROJECT / "outputs/phase16k-colab/cuda_preflight.json"
subprocess.run(
    [str(PYTHON), "scripts/phase16i_cuda_preflight.py", "--output", str(PREFLIGHT)],
    cwd=PROJECT,
    env=ENV,
    check=True,
)
preflight = json.loads(PREFLIGHT.read_text())
print(
    json.dumps(
        {
            key: preflight[key]
            for key in (
                "status",
                "model",
                "model_revision",
                "qlora_kernel_probe",
                "disk_free_gib",
                "blockers",
            )
        },
        indent=2,
    )
)
if preflight["status"] != "ready" or preflight["qlora_kernel_probe"] != "passed":
    raise RuntimeError("CUDA preflight blocked full training. Stop here.")
contract_check = subprocess.run(
    [
        str(PYTHON),
        "-c",
        "import json, runpy; "
        "c, _ = runpy.run_path('scripts/phase16k_full_runner.py')['paired_contract'](); "
        "print(json.dumps(c))",
    ],
    cwd=PROJECT,
    env=ENV,
    check=True,
    capture_output=True,
    text=True,
)
CONTRACT = json.loads(contract_check.stdout)
print("Matched full-run contract:", json.dumps(CONTRACT, indent=2))

## 6. Mount Drive and verify Phase 16J smoke evidence

Set `PERSIST_FULL_TO_DRIVE = True` deliberately before full training. The wrapper mirrors verified checkpoints to a new versioned Drive destination **during** training. Adjust `SMOKE_DRIVE_ROOT` if your Phase 16J artifacts were saved elsewhere. Both smoke results must show 20 completed optimizer steps. This cell refuses an unmounted Drive and an occupied experiment destination.

In [ ]:
PERSIST_FULL_TO_DRIVE = False
DRIVE_ROOT = Path("/content/drive/MyDrive/multilingual-llm-reasoning/phase16k-full-training-v1")
SMOKE_DRIVE_ROOT = Path("/content/drive/MyDrive/multilingual-llm-reasoning/phase16j-smoke-v1")
if PERSIST_FULL_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    assert Path("/content/drive").is_mount(), "Google Drive did not mount"
    for condition in ("english", "multilingual"):
        run_file = SMOKE_DRIVE_ROOT / f"{condition}_smoke/run.json"
        if not run_file.is_file():
            raise FileNotFoundError(f"Phase 16J smoke evidence is missing: {run_file}")
        record = json.loads(run_file.read_text())
        assert record["status"] == "completed"
        assert record["metrics"]["global_step"] == 20
    if DRIVE_ROOT.exists():
        raise FileExistsError(
            f"Full-training Drive root already exists: {DRIVE_ROOT}. "
            "Inspect it; use the recovery procedure if this is an interrupted run."
        )
    print("Smoke evidence verified; new Drive destination:", DRIVE_ROOT)
else:
    print("Drive persistence is disabled. Full-training cells will refuse to start.")

## 7. English-only full training — 75 optimizer steps

Enable only after all checks pass. The runner refuses existing local/Drive outputs and launches the existing `train --mode full` in a fresh subprocess. It initializes a fresh LoRA adapter on the pinned base, never from a smoke checkpoint. Checkpoints at steps 25, 50, and 75 are copied to Drive as they become complete.

In [ ]:
RUN_ENGLISH_FULL = False
ENGLISH_LOCAL = PROJECT / "outputs/training/phase16i_english_lora/full"
if RUN_ENGLISH_FULL:
    if not PERSIST_FULL_TO_DRIVE or preflight["status"] != "ready":
        raise RuntimeError("Enable verified Drive persistence and pass CUDA preflight first")
    subprocess.run(
        [
            str(PYTHON),
            "scripts/phase16k_full_runner.py",
            "--condition",
            "english",
            "--drive-root",
            str(DRIVE_ROOT),
            "--confirm-full-training",
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
    )
else:
    print("English full training pending; set RUN_ENGLISH_FULL=True deliberately.")

## 8. Verify English checkpoint and adapter

The runner itself checks exact step 75, finite losses, complete checkpoint and adapter files, fresh-base adapter activation, a tiny forward pass, dataset hashes, and Drive copy hashes. This cell reads those actual artifacts; it does not rerun training.

In [ ]:
ENGLISH_FINAL = DRIVE_ROOT / "english/final"
if ENGLISH_FINAL.is_dir():
    english_completion = json.loads((ENGLISH_FINAL / "completion.json").read_text())
    english_adapter_check = json.loads(
        (ENGLISH_FINAL / "adapter_load_verification.json").read_text()
    )
    assert english_completion["status"] == "completed"
    assert english_completion["global_step"] == 75
    assert english_adapter_check["status"] == "passed"
    print("English full run verified:", ENGLISH_FINAL)
else:
    print("English full run pending; no completion claim.")

## 9. Multilingual full training — independent 75 optimizer steps

Run only after English verification. Because the trainer is a child process, its model and optimizer are released on exit before this second condition starts. The wrapper requires a separate empty output directory and creates a fresh adapter from the same pinned base.

In [ ]:
RUN_MULTILINGUAL_FULL = False
MULTILINGUAL_LOCAL = PROJECT / "outputs/training/phase16i_multilingual_lora/full"
if RUN_MULTILINGUAL_FULL:
    if not PERSIST_FULL_TO_DRIVE or preflight["status"] != "ready":
        raise RuntimeError("Enable verified Drive persistence and pass CUDA preflight first")
    if not ENGLISH_FINAL.is_dir():
        raise RuntimeError("Verify English full training before the multilingual run")
    subprocess.run(
        [
            str(PYTHON),
            "scripts/phase16k_full_runner.py",
            "--condition",
            "multilingual",
            "--drive-root",
            str(DRIVE_ROOT),
            "--confirm-full-training",
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
    )
else:
    print("Multilingual full training pending; set RUN_MULTILINGUAL_FULL=True deliberately.")

## 10. Verify multilingual checkpoint and adapter

In [ ]:
MULTILINGUAL_FINAL = DRIVE_ROOT / "multilingual/final"
if MULTILINGUAL_FINAL.is_dir():
    multilingual_completion = json.loads((MULTILINGUAL_FINAL / "completion.json").read_text())
    multilingual_adapter_check = json.loads(
        (MULTILINGUAL_FINAL / "adapter_load_verification.json").read_text()
    )
    assert multilingual_completion["status"] == "completed"
    assert multilingual_completion["global_step"] == 75
    assert multilingual_adapter_check["status"] == "passed"
    print("Multilingual full run verified:", MULTILINGUAL_FINAL)
else:
    print("Multilingual full run pending; no completion claim.")

## 11. Compare training results and Drive artifacts

This reads only actual training and designated validation metrics; it does not use the held-out multilingual test set. Equal optimizer-step and example budgets are checked. The multilingual prompt-token total is about 3.18% higher and is reported without adjusting the budget.

In [ ]:
RESULTS = []
for condition, final in (("english", ENGLISH_FINAL), ("multilingual", MULTILINGUAL_FINAL)):
    if not final.is_dir():
        print(condition, "PENDING")
        continue
    record = json.loads((final / "completion.json").read_text())
    metrics = record["metrics"]
    assert record["global_step"] == metrics["global_step"] == 75
    assert (
        DRIVE_ROOT / condition / "checkpoints/checkpoint-75/checkpoint_integrity.json"
    ).is_file()
    prompt_per_epoch = 23113 if condition == "english" else 24669
    response_per_epoch = 25870
    log_rows = [
        json.loads(line)
        for line in (final / "training_log.jsonl").read_text().splitlines()
        if line.strip()
    ]
    last_learning_rate = next(
        (row["learning_rate"] for row in reversed(log_rows) if "learning_rate" in row),
        None,
    )
    RESULTS.append(
        {
            "condition": condition,
            "optimizer_steps": 75,
            "training_examples": 197,
            "effective_batch_size": 8,
            "seed": 42,
            "configured_learning_rate": 0.0002,
            "last_logged_learning_rate": last_learning_rate,
            "prompt_tokens_per_epoch": prompt_per_epoch,
            "supervised_response_tokens_per_epoch": response_per_epoch,
            "nominal_total_tokens_three_epochs": 3 * (prompt_per_epoch + response_per_epoch),
            "train_loss": metrics.get("train_loss"),
            "validation_loss": metrics.get("eval_loss"),
            "runtime_seconds": metrics.get("runtime_seconds"),
            "peak_allocated_gib": metrics.get("peak_allocated_gib"),
            "peak_reserved_gib": metrics.get("peak_reserved_gib"),
            "checkpoint": str(DRIVE_ROOT / condition / "checkpoints/checkpoint-75"),
            "adapter": str(final / "adapter"),
        }
    )
print(json.dumps(RESULTS, indent=2))
if len(RESULTS) != 2:
    print("Both completed CUDA runs are required for a comparison.")

## 12. Verify persistence and write a completion report

Only when both runs are complete, verify the Drive adapter and checkpoint files are readable, preserve the environment freeze, and write a versioned completion report without overwriting a previous one. No test-set evaluation is run.

In [ ]:
if len(RESULTS) == 2:
    for condition in ("english", "multilingual"):
        final = DRIVE_ROOT / condition / "final"
        for rel in (
            "adapter/adapter_model.safetensors",
            "adapter/adapter_config.json",
            "metrics.json",
            "run.json",
            "adapter_load_verification.json",
        ):
            path = final / rel
            assert path.is_file() and path.stat().st_size > 0, path
        checkpoint = DRIVE_ROOT / condition / "checkpoints/checkpoint-75"
        assert (checkpoint / "checkpoint_integrity.json").is_file()
    freeze_copy = DRIVE_ROOT / "environment.freeze.txt"
    assert freeze_copy.is_file(), "Environment record was not persisted by the runner"
    assert (
        hashlib.sha256(freeze_copy.read_bytes()).digest()
        == hashlib.sha256((PROJECT / ".venv-colab/environment.freeze.txt").read_bytes()).digest()
    )
    report_path = DRIVE_ROOT / "phase16k_training_completion_report.md"
    if report_path.exists():
        raise FileExistsError(f"Refusing to overwrite report: {report_path}")
    lines = [
        "# Phase 16K full-training completion",
        "",
        "Both CUDA conditions completed 75 optimizer steps.",
        "",
        f"Pinned base: `{CONTRACT['model']}` at `{CONTRACT['revision']}`.",
        "",
        (
            "The datasets remained frozen and matched by 197 canonical IDs. "
            "The multilingual set has 3.18% more prompt tokens."
        ),
        "",
        "| Condition | Train loss | Validation loss | Runtime (s) | Peak reserved (GiB) |",
        "|---|---:|---:|---:|---:|",
    ]
    for row in RESULTS:
        lines.append(
            f"| {row['condition']} | {row['train_loss']} | "
            f"{row['validation_loss']} | {row['runtime_seconds']} | "
            f"{row['peak_reserved_gib']} |"
        )
    lines.extend(["", "Held-out multilingual evaluation was not run during training.", ""])
    report_path.write_text("\n".join(lines))
    print("Completion report:", report_path)
else:
    print("Completion report pending both verified full runs.")

## Recovery after a Colab disconnect

Do not rerun a completed full-training cell. Use the separate recovery procedure in `docs/training/phase16k_checkpoint_recovery.md`. It verifies a Drive checkpoint and exact training identity before calling the existing trainer with `--resume latest`, and refuses step 75 as a resume source. The 75-step ceiling is fixed by the unchanged full config.